# 🚁 YOLOv8s Drone Detection (Tesla P100 Optimized)

**Hardware:** Tesla P100 (16GB VRAM)  
**Dataset:** Drone-Detection Multi-Class  
**Goal:** Parallel training with SSD MobileNet

---

### 1. Setup Environment

In [1]:
!pip install -q ultralytics
import os
import yaml
import glob
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

print("✅ Setup Complete")

✅ Setup Complete


### 2. Locate Dataset & Fix Paths

In [2]:
dataset_root = '/kaggle/input/datasets/cybersimar08/drone-detection/drone-detection-new.v5-new-train.yolov8'
new_yaml_path = '/kaggle/working/drone_data_p100.yaml'

with open(f'{dataset_root}/data.yaml', 'r') as f:
    data_config = yaml.safe_load(f)

data_config['path'] = dataset_root
data_config['train'] = 'train/images'
data_config['val'] = 'valid/images'
data_config['test'] = 'test/images'

with open(new_yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print(f"✅ Dataset config fixed at: {new_yaml_path}")
class_names = data_config['names']
print(f"📋 Classes: {class_names}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/cybersimar08/drone-detection/drone-detection-new.v5-new-train.yolov8/data.yaml'

### 3. Visual Dataset Check
This cell displays a few images from the training set to verify the labels are correct.

In [ ]:
train_images = glob.glob(f"{dataset_root}/train/images/*.jpg")[:3]
plt.figure(figsize=(15, 10))

for i, img_path in enumerate(train_images):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(1, 3, i+1)
    plt.imshow(img)
    plt.title(f"Sample {i+1}")
    plt.axis('off')

plt.show()
print("👁️ Dataset preview displayed above.")

### 4. Train YOLOv8s (P100 Optimized)

In [ ]:
model = YOLO('yolov8s.pt') 

results = model.train(
    data=new_yaml_path,
    epochs=30,
    imgsz=640,
    batch=32,             # Optimized for P100 16GB VRAM
    device=0,
    workers=4,
    cache=True,           # RAM Caching for faster epochs
    amp=True,
    project='/kaggle/working/runs',
    name='yolov8s_p100_drone'
)

5. MODEL EVAL

In [ ]:
# Cell 5: Model Evaluation (mAP, Precision, Recall)
print("📊 Evaluating model on validation set...")

# Run validation
metrics = model.val()

print(f'\n📈 Overall Performance:')
print(f'  mAP50:     {metrics.box.map50:.4f}')
print(f'  mAP50-95:  {metrics.box.map:.4f}')
print(f'  Precision: {metrics.box.mp:.4f}')
print(f'  Recall:    {metrics.box.mr:.4f}')

# Per-class metrics
print(f'\n📊 Per-Class mAP50:')
for i, name in enumerate(class_names):
    print(f'  {name}: {metrics.box.maps[i]:.4f}')

5. Prediction

In [ ]:
# Cell 6: Test Predictions on Random Images
import glob
import cv2
import random
import matplotlib.pyplot as plt

# Path to validation images on Kaggle
val_dir = f'{dataset_root}/valid/images'
all_imgs = glob.glob(f'{val_dir}/*')

# Select 6 random images
test_imgs = random.sample(all_imgs, min(6, len(all_imgs)))

# Run prediction (device=0 for P100 GPU)
results = model.predict(test_imgs, conf=0.25, device=0, verbose=False)

# Setup the 2x3 grid
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, result in enumerate(results):
    # Get the image with bounding boxes drawn
    img_bgr = result.plot()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    axes[idx].imshow(img_rgb)
    axes[idx].axis('off')
    
    # Count detections per class for the title
    detections = {}
    for box in result.boxes:
        cls_id = int(box.cls[0])
        cls_name = class_names[cls_id]
        detections[cls_name] = detections.get(cls_name, 0) + 1
    
    title = ', '.join([f'{k}:{v}' for k,v in detections.items()]) if detections else 'No detections'
    axes[idx].set_title(title, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()
print('✓ Predictions displayed successfully')